# DATA STORIES – EXPLORING GLOBAL URBAN TREE COVERAGE
## Transforming Urban Data into Meaningful Insights with pandas.pivot_table()

**Student:** NIDEESH C G  
**Register Number:** 23MIC0092  
**Course:** CSI3007 – Advanced Python Programming  
**Degree:** Integrated M.Tech Computer Science and Engineering (Data Science)  

> **Academic Transparency Notice:**  
> *Reconstructed Demonstration – Recreated from the original course activity using the cited data source. Original submission files were unavailable.*

---
### Source Attribution
- **Article:** Visual Capitalist – *Ranked: The World’s Cities With the Most Tree Coverage*  
- **URL:** https://www.visualcapitalist.com/ranked-the-worlds-cities-with-the-most-tree-coverage/  
- **Primary Data:** MIT Senseable City Lab Treepedia, in collaboration with the World Economic Forum  
- **Key Metric:** Green View Index (GVI %), estimating pedestrian street-level tree canopy visibility using computer vision on Google Street View panoramas.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)

## 1. Data Collection & Ingestion
We load the verified raw dataset containing 34 world cities, their country codes, and raw Green View Index percentages.

In [2]:
raw_path = 'urban_tree_coverage_data.csv'
if not os.path.exists(raw_path):
    raw_path = 'public/assets/pivot-table/urban_tree_coverage_data.csv'

df_raw = pd.read_csv(raw_path)
df_raw.head(10)

## 2. Data Preprocessing & Feature Engineering
1. Strip percentage symbols and convert to numerical float.
2. Map country codes to full country names and geographic continents.
3. Segment cities into Canopy Tiers: `High (>25%)`, `Moderate (15-25%)`, and `Low (<15%)`.

In [3]:
df = df_raw.copy()
df['City'] = df['City_Raw']
df['Green_View_Index_Pct'] = df['Green_View_Index_Raw'].str.rstrip('%').astype(float)

country_map = {
    'US': ('United States', 'North America'),
    'NL': ('Netherlands', 'Europe'),
    'SG': ('Singapore', 'Asia'),
    'NO': ('Norway', 'Europe'),
    'AU': ('Australia', 'Oceania'),
    'CA': ('Canada', 'North America'),
    'MX': ('Mexico', 'Latin America'),
    'ZA': ('South Africa', 'Africa'),
    'DE': ('Germany', 'Europe'),
    'CH': ('Switzerland', 'Europe'),
    'IL': ('Israel', 'Asia'),
    'IT': ('Italy', 'Europe'),
    'AR': ('Argentina', 'South America'),
    'BR': ('Brazil', 'South America'),
    'GB': ('United Kingdom', 'Europe'),
    'EC': ('Ecuador', 'South America'),
    'JP': ('Japan', 'Asia'),
    'FR': ('France', 'Europe')
}

df['Country'] = df['Country_Code'].map(lambda c: country_map[c][0])
df['Continent'] = df['Country_Code'].map(lambda c: country_map[c][1])

def get_tier(val):
    if val >= 25.0:
        return 'High (>25%)'
    elif val >= 15.0:
        return 'Moderate (15-25%)'
    else:
        return 'Low (<15%)'

df['Canopy_Tier'] = df['Green_View_Index_Pct'].apply(get_tier)
df[['Rank', 'City', 'Country', 'Continent', 'Green_View_Index_Pct', 'Canopy_Tier']].head()

## 3. Pivot Table Analysis using `pandas.pivot_table()`
We implement multidimensional aggregations using `pandas.pivot_table()` to uncover continental and national patterns.

In [4]:
# Pivot Table 1: Continental Multi-Metric Aggregation
pivot_cont = pd.pivot_table(
    df,
    index='Continent',
    values='Green_View_Index_Pct',
    aggfunc=['count', 'mean', 'median', 'max', 'min', 'std']
)
pivot_cont.columns = ['Count', 'Mean GVI (%)', 'Median GVI (%)', 'Max GVI (%)', 'Min GVI (%)', 'Std Dev (%)']
pivot_cont.sort_values('Mean GVI (%)', ascending=False).round(2)

In [5]:
# Pivot Table 2: Cross-tabulation of Continents vs Canopy Tiers
pivot_tiers = pd.pivot_table(
    df,
    index='Continent',
    columns='Canopy_Tier',
    values='City',
    aggfunc='count',
    fill_value=0
)
pivot_tiers

## 4. Visualizations
Visualizing the ranked street canopy across all 34 world cities using the black-and-orange theme.

In [6]:
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 8))
df_sorted = df.sort_values('Green_View_Index_Pct')
ax.barh(df_sorted['City'], df_sorted['Green_View_Index_Pct'], color='#FF6500')
ax.set_title('Global Cities Ranked by Green View Index (%)')
ax.set_xlabel('GVI (%)')
plt.tight_layout()
plt.show()

## 5. Analytical Findings & Academic Conclusions
1. **Tampa Leads Globally (36.1%):** Proactive municipal tree planting and robust tree protection codes ensure exceptional street-level canopy presence.
2. **Street-Level vs Satellite Discrepancy:** High-density cities like Paris (8.8%) may feature grand boulevards and distant peripheral woods, but have lower pedestrian-level street greenness compared to cities designed around tree-lined residential corridors.
3. **North American Consistency:** 5 of the top 12 world cities reside in North America, highlighting modern zoning that incorporates pedestrian street shade.

---
**Course:** CSI3007 Advanced Python Programming  
**Student:** Nideesh C G (23MIC0092)